# 🎈 Agesis EYE — Stage 03: YOLOv8 Black Balloon Model Training

This notebook trains a high-precision, low-latency **YOLOv8 Nano** model to detect black balloons from an ESP32-CAM stream.

### 🎯 Objectives & Rules
- **Target Class**: `0: balloon`
- **Architecture**: `yolov8n.pt` (fine-tuned from pretrained COCO weights)
- **Resolution**: `imgsz=320` (matches 320x240 QVGA camera stream 1:1)
- **Precision Target**: $\ge 0.97$ (crucial for safe laser targeting — zero false fires)
- **Recall Target**: $\ge 0.90$
- **Export**: `best.onnx` for lightweight, real-time laptop inference (Stage 04)

In [1]:
# 1. Verify GPU Acceleration & Install Ultralytics
!nvidia-smi
!pip install ultralytics -q

import os
import sys
import glob
import yaml
import torch
from IPython.display import Image, display
from ultralytics import YOLO

print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name : {torch.cuda.get_device_name(0)}")

Thu Oct  1 19:52:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Dataset Setup & Path Resolution

Automatically finds `data.yaml` whether running on:
1. **Kaggle** (`/kaggle/input/...`)
2. **Google Colab** (`/content/...`)
3. **Local Machine**

In [ ]:
# Auto-detect data.yaml location across all environments (Kaggle / Colab / Local)
import os, glob, subprocess, yaml

def find_file_recursive(target_name, search_dirs):
    for d in search_dirs:
        if os.path.isdir(d):
            for root, dirs, files in os.walk(d):
                if target_name in files:
                    return os.path.abspath(os.path.join(root, target_name))
    return None

def find_zips_recursive(search_dirs):
    found = []
    for d in search_dirs:
        if os.path.isdir(d):
            for root, dirs, files in os.walk(d):
                for f in files:
                    if f.lower().endswith('.zip') and not f.startswith('.'):
                        found.append(os.path.join(root, f))
    return found

search_dirs = ['/kaggle/input', '/kaggle/working', '/content', 'dataset', '.', '..']

# 1. Search for existing data.yaml
yaml_path = find_file_recursive('data.yaml', search_dirs)

# 2. If data.yaml not found, look for zip files and extract them
if yaml_path is None:
    zip_candidates = find_zips_recursive(search_dirs)
    if zip_candidates:
        print(f'[+] Found dataset archive: {zip_candidates[0]}')
        print(f'[+] Extracting to ./dataset ...')
        os.makedirs('dataset', exist_ok=True)
        subprocess.run(['unzip', '-q', '-o', zip_candidates[0], '-d', 'dataset'], check=True)
        yaml_path = find_file_recursive('data.yaml', ['dataset', '.'])

# 3. Verify yaml_path was found
if yaml_path is None:
    print('\n' + '='*70)
    print('❌ ERROR: data.yaml not found in /kaggle/input or local directories!')
    print('='*70)
    print('\n[Diagnostic] /kaggle/input contents:')
    if os.path.exists('/kaggle/input'):
        items = list(os.walk('/kaggle/input'))
        if not items or (len(items[0][1]) == 0 and len(items[0][2]) == 0):
            print('  /kaggle/input is currently EMPTY.')
        else:
            for root, dirs, files in items[:10]:
                print(f'  {root}:')
                for f in files[:8]:
                    print(f'    - {f}')
    else:
        print('  /kaggle/input does not exist.')
    print('\n👉 HOW TO FIX ON KAGGLE:')
    print('  1. Look at the right sidebar in your Kaggle Notebook under "Input".')
    print('  2. Click "+ Add Input" (or "+ Add Data").')
    print('  3. Select your uploaded "agesis-balloon-dataset" OR upload "agesis_balloon_dataset_kaggle.zip".')
    print('  4. Once attached, re-run this cell!')
    print('='*70 + '\n')
    raise FileNotFoundError('data.yaml could not be found. Please attach the dataset via + Add Input in Kaggle.')

print(f'[+] Found data.yaml at: {yaml_path}')
dataset_root = os.path.dirname(yaml_path)

with open(yaml_path, 'r') as f:
    data_cfg = yaml.safe_load(f)

# Ensure absolute path points directly to dataset root directory
data_cfg['path'] = dataset_root
working_yaml = os.path.abspath('data_train.yaml')
with open(working_yaml, 'w') as f:
    yaml.dump(data_cfg, f, default_flow_style=False, sort_keys=False)

print(f'[+] Configured training YAML: {working_yaml}')
with open(working_yaml) as f:
    print(f.read())


## 3. Train YOLOv8 Nano Model

- Fine-tuning from pretrained COCO weights (`yolov8n.pt`)
- `imgsz=320` to match camera stream directly
- `patience=25` early stopping
- `fliplr=0.5` horizontal reflection, `flipud=0.0` (balloons don't float upside-down)
- `mosaic=1.0` for multi-scale robustness

In [ ]:
# Load pretrained model
model = YOLO("yolov8n.pt")

# Train
results = model.train(
    data=working_yaml,
    epochs=80,
    imgsz=320,
    batch=16,
    patience=25,
    device=0 if torch.cuda.is_available() else 'cpu',
    workers=4,
    optimizer='auto',
    lr0=0.01,
    lrf=0.01,
    fliplr=0.5,
    flipud=0.0,
    mosaic=1.0,
    project='runs',
    name='agesis_balloon_v1',
    plots=True,
    verbose=True
)

print("\n[+] Training finished successfully!")

## 4. Evaluate on Held-out Test Split

Stage 03 Rule: **Judge solely by the held-out test split, never training metrics.**

In [ ]:
# Validate on held-out TEST set
best_weights = "runs/agesis_balloon_v1/weights/best.pt"
eval_model = YOLO(best_weights)

metrics = eval_model.val(data=working_yaml, split='test')

precision = metrics.box.mp
recall = metrics.box.mr
map50 = metrics.box.map50
map50_95 = metrics.box.map

print(f"\n{'=' * 45}")
print(f"  HELD-OUT TEST SET EVALUATION")
print(f"{'=' * 45}")
print(f"  Precision (target >= 0.97) : {precision:.4f} {'[PASS]' if precision >= 0.97 else '[REVIEW]'}")
print(f"  Recall    (target >= 0.90) : {recall:.4f} {'[PASS]' if recall >= 0.90 else '[REVIEW]'}")
print(f"  mAP@50                     : {map50:.4f}")
print(f"  mAP@50-95                  : {map50_95:.4f}")
print(f"{'=' * 45}")

## 5. Visual Inspection of Training Curves & Confusion Matrix

In [ ]:
# Display Confusion Matrix & Loss Plots
plots_to_show = [
    "runs/agesis_balloon_v1/results.png",
    "runs/agesis_balloon_v1/confusion_matrix.png",
    "runs/agesis_balloon_v1/val_batch0_pred.jpg"
]

for p in plots_to_show:
    if os.path.exists(p):
        print(f"\n--- {os.path.basename(p)} ---")
        display(Image(filename=p))

## 6. Export to ONNX for Laptop Inference (Stage 04)

ONNX allows ultra-fast CPU inference on your laptop without requiring PyTorch/CUDA runtime.

In [ ]:
# Export best model to ONNX
onnx_path = eval_model.export(format="onnx", imgsz=320, dynamic=False, simplify=True)
print(f"[+] Exported ONNX model: {onnx_path}")

# Package weights for download
!zip -j agesis_model_weights.zip runs/agesis_balloon_v1/weights/best.pt runs/agesis_balloon_v1/weights/best.onnx

print(f"\n{'=' * 50}")
print(f"  Model training & export complete!")
print(f"  Download 'agesis_model_weights.zip' to use in Stage 04")
print(f"{'=' * 50}")